In [5]:
from typing import Optional
from datetime import datetime
import pandas as pd
import numpy as np
!pip install nfl_data_py==0.3.3
import nfl_data_py as nfl

pd.set_option('display.max_columns', None)

# from data.schema.database import (
#     Player, NFLSeasonStats, NFLAdvancedStats,
#     NFLWeeklySnaps, InjuryRecord, get_session
# )

# Positions we care about for fantasy
SKILL_POSITIONS = {"QB", "RB", "WR", "TE"}

# Injury severity mapping — used for composite injury risk score
INJURY_SEVERITY = {
    "Knee": 0.8, "ACL": 1.0, "MCL": 0.7, "Meniscus": 0.7,
    "Hamstring": 0.6, "Quad": 0.5, "Groin": 0.5, "Hip": 0.5,
    "Ankle": 0.6, "Foot": 0.5,
    "Shoulder": 0.6, "Clavicle": 0.5, "Elbow": 0.4, "Wrist": 0.4,
    "Back": 0.7, "Ribs": 0.5,
    "Concussion": 0.8,
    "Illness": 0.2, "Rest": 0.0, "Not Injury Related": 0.0,
}

# Soft tissue injuries have higher recurrence risk
SOFT_TISSUE_INJURIES = {"Hamstring", "Quad", "Groin", "Hip", "Calf", "Thigh"}

# from utils.constants import COACHING_DATA, _COACHING_FALLBACK, TEAM_FULL_NAMES


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
years = [2020]

ids = nfl.import_ids()
player_ids = ids[['gsis_id', 'pfr_id']][~ids['gsis_id'].isna()].rename(columns={'gsis_id': 'player_id', 'pfr_id': 'pfr_player_id'}).drop_duplicates()

In [6]:
def run_full_backfill(start_year: int = 2015, end_year: Optional[int] = None):
        """
        Master method — runs all ingestion steps in order.
        Pulls everything from start_year to present.
        Safe to re-run (upserts, not inserts).
        """
        if not end_year:
            end_year = datetime.now().year
        years = list(range(start_year, end_year + 1))

        print(f"Starting NFL backfill for {years[0]}–{years[-1]}")

        ingest_players()
        ingest_seasonal_stats(years)
        ingest_advanced_stats(years)
        ingest_snap_counts(years)
        ingest_injuries(years)

        print("NFL backfill complete.")

def _load_snap_pct_for_season_stats(years: list[int]) -> pd.DataFrame:
        """Aggregate snap % per player per season from weekly snap data."""
        try:
            snaps = nfl.import_snap_counts(years)
            # print(snaps.columns)
            # print(player_ids.columns)
            snaps = player_ids.merge(snaps, on=["pfr_player_id"], how="inner")
            snap_agg = (
                snaps[snaps["position"].isin(SKILL_POSITIONS)]
                .groupby(["player_id", "team", "season"])
                .agg(snap_pct=("offense_pct", "mean"))
                .reset_index()
            )
            return snap_agg
        except Exception as e:
            logger.warning(f"Could not load snap data: {e}")
            return pd.DataFrame()
    
def _load_team_targets(years: list[int]) -> pd.DataFrame:
    """Compute team-level target totals (needed for target_share)."""
    try:
        pbp = nfl.import_pbp_data(years, columns=[
            "passer_player_id", "receiver_player_id", "pass_attempt",
            "posteam", "season", "play_type", "game_id"
        ])
        pass_plays = pbp[pbp["pass_attempt"] == 1]
        team_targets = (
            pass_plays.groupby(["posteam", "season"])
            .size()
            .reset_index(name="team_target_total")
            .rename(columns={"posteam": "team"})
        )
        return team_targets
    except Exception as e:
        print(f"Could not load team targets: {e}")
        return pd.DataFrame()
            
def ingest_seasonal_stats(years: list[int]):
        """
        Pulls seasonal aggregates: standard stats + fantasy points + efficiency.
        This is the core stats table.
        """
        print(f"Ingesting seasonal stats for {years}...")

        player_stats_df = nfl.import_seasonal_data(years, s_type="REG")
        # print("RAW DATA")
        # print(type(player_stats_df))
        # print(player_stats_df.columns)
        snap_df = _load_snap_pct_for_season_stats(years)
        # print("SNAP DATA")
        # print(type(snap_df))
        # print(snap_df.columns)
        team_targets = _load_team_targets(years)
        # print("TEAM TARGETS DATA")
        # print(type(team_targets))
        # print(team_targets.columns)
        
        # Merge snap pct and team targets into seasonal
        if not snap_df.empty:
            player_stats_df = player_stats_df.merge(snap_df, on=["player_id", "season"], how="left")
        if not team_targets.empty:
            player_stats_df = player_stats_df.merge(team_targets, on=["season"], how="left")

        # Compute derived metrics not in raw data
        player_stats_df["wopr"] = (1.5 * player_stats_df.get("target_share", 0)) + (0.7 * player_stats_df.get("air_yards_share", 0))
        player_stats_df["tgt_per_game"] = player_stats_df["targets"] / player_stats_df["games"].replace(0, float("nan"))
        player_stats_df["catch_rate"] = player_stats_df["receptions"] / player_stats_df["targets"].replace(0, float("nan"))
        player_stats_df["fantasy_ppg_ppr"] = player_stats_df["fantasy_points_ppr"] / player_stats_df["games"].replace(0, float("nan"))
        player_stats_df["fantasy_ppg_half"] = (player_stats_df.get("fantasy_points", player_stats_df["fantasy_points_ppr"] * 0.9)) / player_stats_df["games"].replace(0, float("nan"))
        player_stats_df["completion_pct"] = player_stats_df["completions"] / player_stats_df["attempts"].replace(0, float("nan"))

        records = []
        return player_stats_df
        # df = pd.DataFrame()
        # for _, row in player_stats_df.iterrows():
        #     # break
        #     print(row.columns)

        #     df_row = pd.DataFrame(
        #         [
        #             dict(
        #                 player_id=str(row["player_id"]),
        #                 season=int(row["season"]),
        #                 season_type="REG",
        #                 team=row.get("recent_team"),
        #                 games=_safe_int(row.get("games")),
        #                 games_started=_safe_int(row.get("games_started")),
        #                 completions=_safe_int(row.get("completions")),
        #                 attempts=_safe_int(row.get("attempts")),
        #                 passing_yards=_safe_int(row.get("passing_yards")),
        #                 passing_tds=_safe_int(row.get("passing_tds")),
        #                 interceptions=_safe_int(row.get("interceptions")),
        #                 passing_epa=row.get("passing_epa"),
        #                 completion_pct=row.get("completion_pct"),
        #                 yards_per_attempt=row.get("yards_per_attempt"),
        #                 passer_rating=row.get("passer_rating"),
        #                 sacks=_safe_int(row.get("sacks")),
        #                 carries=_safe_int(row.get("carries")),
        #                 rushing_yards=_safe_int(row.get("rushing_yards")),
        #                 rushing_tds=_safe_int(row.get("rushing_tds")),
        #                 rushing_epa=row.get("rushing_epa"),
        #                 yards_per_carry=row.get("rushing_yards_per_att"),
        #                 targets=_safe_int(row.get("targets")),
        #                 receptions=_safe_int(row.get("receptions")),
        #                 receiving_yards=_safe_int(row.get("receiving_yards")),
        #                 receiving_tds=_safe_int(row.get("receiving_tds")),
        #                 receiving_epa=row.get("receiving_epa"),
        #                 yards_per_reception=row.get("yards_per_reception"),
        #                 catch_rate=row.get("catch_rate"),
        #                 yards_per_target=row.get("yards_per_target"),
        #                 air_yards_total=_safe_int(row.get("receiving_air_yards")),
        #                 yards_after_catch=_safe_int(row.get("receiving_yards_after_catch")),
        #                 fantasy_points_ppr=row.get("fantasy_points_ppr"),
        #                 fantasy_ppg_ppr=row.get("fantasy_ppg_ppr"),
        #                 target_share=row.get("target_share"),
        #                 air_yards_share=row.get("air_yards_share"),
        #                 racr=row.get("racr"),
        #                 wopr=row.get("wopr"),
        #                 tgt_per_game=row.get("tgt_per_game"),
        #                 snap_pct=row.get("snap_pct")
        #             )
        #         ]
        #     )
            
        #     df = pd.concat([df, df_row], ignore_index=True)

        # self._bulk_upsert(NFLSeasonStats, records, conflict_column=("player_id", "season", "season_type"))
    
        print(f"{len(records)} seasonal stat rows ingested.")
        return df

def ingest_snap_counts(years: list[int]):
        """
        Weekly offensive snap counts. Useful for tracking role stability
        and detecting depth chart movement mid-season.
        """
        print(f"Ingesting weekly snap counts for {years}...")
        try:
            raw = nfl.import_snap_counts(years)
            raw = raw[raw["position"].isin(SKILL_POSITIONS)].copy()
        except Exception as e:
            print(f"Snap counts failed: {e}")
            return

        records = [
            NFLWeeklySnaps(
                player_id=str(row.get("pfr_player_id", "")),
                season=int(row["season"]),
                week=int(row["week"]),
                game_id=row.get("game_id", ""),
                team=row.get("team"),
                offense_snaps=_safe_int(row.get("offense_snaps")),
                offense_pct=row.get("offense_pct"),
                defense_snaps=_safe_int(row.get("defense_snaps")),
                st_snaps=_safe_int(row.get("st_snaps")),
            )
            for _, row in raw.iterrows()
        ]

        # self._bulk_upsert(NFLWeeklySnaps, records, conflict_column=None)  # no unique constraint here
        print(f"{len(records)} snap count rows ingested.")


def ingest_advanced_stats(years: list[int]):
    """
    Pull Next Gen Stats tracking data for all three stat types.
    Only available from 2016+.
    """
    ngs_years = [y for y in years if y >= 2016]
    if not ngs_years:
        return

    logger.info(f"Ingesting NGS advanced stats for {ngs_years}...")

    for stat_type in ("passing", "rushing", "receiving"):
        try:
            raw = nfl.import_ngs_data(stat_type, ngs_years)
            records = _parse_ngs(raw, stat_type)
            # self._bulk_upsert(NFLAdvancedStats, records, conflict_column=("player_id", "season", "stat_type"))
            # logger.info(f"  → {len(records)} NGS {stat_type} rows ingested.")
        except Exception as e:
            logger.warning(f"NGS {stat_type} failed: {e}")

def _parse_ngs(raw: pd.DataFrame, stat_type: str):
    """
    Parse NGS dataframe into ORM objects.
    """
    # Filter to full-season rows (week = 0 in nfl_data_py NGS aggregation)
    if "week" in raw.columns:
        raw = raw[raw["week"] == 0].copy()

    records = []
    for _, row in raw.iterrows():
        pid = str(row.get("player_gsis_id") or row.get("player_id", ""))
        if not pid:
            continue

        # r = NFLAdvancedStats(
        #     player_id=pid,
        #     season=int(row["season"]),
        #     stat_type=stat_type,
        # )
        r = dict(
            player_id=pid,
            season=int(row["season"]),
            stat_type=stat_type,
        )
        if stat_type == "passing":
            r['avg_time_to_throw'] = row.get("avg_time_to_throw")
            r['avg_completed_air_yards'] = row.get("avg_completed_air_yards")
            r['avg_intended_air_yards'] = row.get("avg_intended_air_yards")
            r['aggressiveness'] = row.get("aggressiveness")
            r['completion_pct_above_expectation'] = row.get("completion_percentage_above_expectation")
            r['avg_air_yards_to_sticks'] = row.get("avg_air_yards_to_sticks")
        elif stat_type == "rushing":
            r['efficiency'] = row.get("efficiency")
            r['avg_time_to_los'] = row.get("avg_time_to_los")
            r['expected_yards'] = row.get("expected_rush_yards")
            r['rush_yards_over_expected'] = row.get("rush_yards_over_expected")
            r['rush_yards_over_expected_per_att'] = row.get("rush_yards_over_expected_per_att")
            r['percent_attempts_gte_eight_defenders'] = row.get("percent_attempts_gte_eight_defenders")
        elif stat_type == "receiving":
            r['avg_cushion'] = row.get("avg_cushion")
            r['avg_separation'] = row.get("avg_separation")
            r['avg_intended_air_yards_rec'] = row.get("avg_intended_air_yards")
            r['catch_pct_above_expectation'] = row.get("catch_percentage_above_expectation")
            r['avg_yac_above_expectation'] = row.get("avg_yac_above_expectation")
        records.append(r)
    return records

In [9]:
raw = nfl.import_ngs_data("receiving", [2020, 2021]).rename(columns={"player_gsis_id": "player_id"})

In [12]:
raw#["catch_pct_above_expectation"]

,season,season_type,week,player_display_name,player_position,team_abbr,avg_cushion,avg_separation,avg_intended_air_yards,percent_share_of_intended_air_yards,receptions,targets,catch_percentage,yards,rec_touchdowns,avg_yac,avg_expected_yac,avg_yac_above_expectation,player_id,player_first_name,player_last_name,player_jersey_number,player_short_name
5860,2020,REG,0,Darnell Mooney,WR,CHI,7.627528,3.219063,11.735408,24.448923,61,98,62.244898,631.0,4,4.420328,3.913009,0.507318,00-0036309,Darnell,Mooney,11,D.Mooney
5861,2020,REG,0,Marquise Brown,WR,BAL,7.455670,3.265745,13.747900,39.296556,58,100,58.000000,769.0,8,4.845345,3.876106,0.969239,00-0035662,Marquise,Brown,15,M.Brown
5862,2020,REG,0,Braxton Berrios,WR,NYJ,7.141667,3.423246,7.708000,10.428849,37,55,67.272727,394.0,3,6.172703,5.614924,0.557779,00-0034419,Braxton,Berrios,10,B.Berrios
5863,2020,REG,0,Laviska Shenault,WR,JAX,7.064571,3.311376,6.410253,10.522958,58,79,73.417722,600.0,5,5.222069,4.135107,1.086962,00-0036268,Laviska,Shenault,10,L.Shenault Jr.
5864,2020,REG,0,Henry Ruggs,WR,LV,6.963590,3.347403,16.919535,16.758382,26,43,60.465116,452.0,2,5.998077,5.830800,0.167277,00-0036357,Henry,Ruggs,11,H.Ruggs
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8950,2021,POST,23,Tee Higgins,WR,CIN,8.247143,1.499096,12.328571,40.053838,4,7,57.142857,100.0,2,10.745000,2.470707,8.274293,00-0036410,Tamaurice,Higgins,85,T.Higgins
8951,2021,POST,23,Ja'Marr Chase,WR,CIN,7.515000,2.726282,10.956250,40.680405,5,8,62.500000,89.0,0,7.536000,7.590445,-0.054445,00-0036900,Ja'Marr,Chase,1,J.Chase
8952,2021,POST,23,Van Jefferson,WR,LAR,6.572500,5.836537,14.086250,32.878191,4,8,50.000000,23.0,0,7.602500,7.331517,0.270983,00-0036415,Vanchii,Jefferson,12,V.Jefferson
8953,2021,POST,23,Ben Skowronek,WR,LAR,5.445000,2.699072,7.024000,10.246535,2,5,40.000000,12.0,0,3.015000,3.679291,-0.664291,00-0036862,Ben,Skowronek,18,B.Skowronek


### Ingest Seasonal Stats

In [174]:
player_stats_df = nfl.import_seasonal_data(years, s_type="REG")
snap_df = _load_snap_pct_for_season_stats(years)
team_targets = _load_team_targets(years)
player_stats_df.columns
if not snap_df.empty:
    player_stats_df = player_stats_df.merge(snap_df, on=["player_id", "season"], how="left")
# if not team_targets.empty:
#     player_stats_df = player_stats_df.merge(team_targets, on=["season", "team"], how="left")

2020 done.
Downcasting floats.


In [180]:
player_stats_deduped_df = player_stats_df.sort_values('snap_pct', ascending=False).groupby(["player_id", "season", "season_type"]).first()
player_stats_deduped_df.reset_index()

,player_id,season,season_type,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr_x,special_teams_tds,fantasy_points,fantasy_points_ppr,games,tgt_sh,ay_sh,yac_sh,wopr_y,ry_sh,rtd_sh,rfd_sh,rtdfd_sh,dom,w8dom,yptmpa,ppr_sh,team,snap_pct
0,00-0019596,2020,REG,401,610,4633.0,40,12.0,21.0,143.0,1,0,5532.0,1810.0,233.0,133.306174,0,13.480107,2.854643,30,6.0,3,3.0,1.0,6.0,-18.186052,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,337.92,337.92,16,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.197091,KC,1.000000
1,00-0020531,2020,REG,275,390,2942.0,24,6.0,13.0,89.0,6,2,2361.0,1478.0,149.0,69.720447,0,15.637096,1.545552,18,-2.0,2,0.0,0.0,4.0,-4.355731,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,209.48,209.48,12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.171110,NO,0.898571
2,00-0022127,2020,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0,0.0,0,0.0,0.0,0.0,0.000000,0,13,17,69.0,2,0.0,0.0,91.0,20.0,8.0,2.407178,0,7.333333,0.561207,0.384410,1.110898,0.0,18.90,31.90,10,0.050296,0.032983,0.014804,0.101830,0.025247,0.095238,0.061069,0.065789,0.060243,0.039245,0.204142,0.032303,LV,0.375000
3,00-0022787,2020,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,3,-4.0,0,0.0,0.0,0.0,0.000000,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,-0.40,-0.40,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.004739,ATL,0.120000
4,00-0022824,2020,REG,1,1,26.0,0,0.0,0.0,0.0,0,0,14.0,12.0,1.0,4.014011,0,1.857143,0.000000,0,0.0,0,0.0,0.0,0.0,0.000000,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,1.04,1.04,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.008076,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629,00-0036433,2020,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,4,24.0,0,0.0,0.0,2.0,1.731275,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,2.40,2.40,1,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,NaN,NaN,0.000000,0.040445,CAR,0.080000
630,00-0036439,2020,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0,0.0,0,0.0,0.0,0.0,0.000000,0,6,9,63.0,0,0.0,0.0,79.0,18.0,4.0,2.915174,0,3.200000,0.241825,0.280559,0.559129,0.0,6.30,12.30,5,0.047368,0.051299,0.025388,0.112092,0.049489,0.000000,0.061538,0.055556,0.024745,0.039592,0.331579,0.027577,DEN,0.108000
631,00-0036442,2020,REG,264,404,2688.0,13,5.0,32.0,231.0,5,3,3429.0,1142.0,150.0,48.966725,0,7.910450,1.200403,37,142.0,3,4.0,1.0,14.0,-0.141221,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,173.72,173.72,10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.190763,CIN,0.973000
632,00-0036450,2020,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,26,109.0,0,0.0,0.0,5.0,-1.833543,0,5,10,34.0,1,1.0,1.0,24.0,23.0,2.0,-3.435261,0,6.658730,0.248911,0.068260,0.421148,0.0,18.30,23

In [179]:
player_stats_deduped_df#.groupby("player_id").size().sort_values(ascending=False)

,,,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr_x,special_teams_tds,fantasy_points,fantasy_points_ppr,games,tgt_sh,ay_sh,yac_sh,wopr_y,ry_sh,rtd_sh,rfd_sh,rtdfd_sh,dom,w8dom,yptmpa,ppr_sh,team,snap_pct
player_id,season,season_type,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
00-0019596,2020,REG,401,610,4633.0,40,12.0,21.0,143.0,1,0,5532.0,1810.0,233.0,133.306174,0,13.480107,2.854643,30,6.0,3,3.0,1.0,6.0,-18.186052,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,337.92,337.92,16,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.197091,KC,1.000000
00-0020531,2020,REG,275,390,2942.0,24,6.0,13.0,89.0,6,2,2361.0,1478.0,149.0,69.720447,0,15.637096,1.545552,18,-2.0,2,0.0,0.0,4.0,-4.355731,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,209.48,209.48,12,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.171110,NO,0.898571
00-0022127,2020,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0,0.0,0,0.0,0.0,0.0,0.000000,0,13,17,69.0,2,0.0,0.0,91.0,20.0,8.0,2.407178,0,7.333333,0.561207,0.384410,1.110898,0.0,18.90,31.90,10,0.050296,0.032983,0.014804,0.101830,0.025247,0.095238,0.061069,0.065789,0.060243,0.039245,0.204142,0.032303,LV,0.375000
00-0022787,2020,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,3,-4.0,0,0.0,0.0,0.0,0.000000,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,-0.40,-0.40,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.004739,ATL,0.120000
00-0022824,2020,REG,1,1,26.0,0,0.0,0.0,0.0,0,0,14.0,12.0,1.0,4.014011,0,1.857143,0.000000,0,0.0,0,0.0,0.0,0.0,0.000000,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,1.04,1.04,1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.008076,None,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
00-0036433,2020,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,4,24.0,0,0.0,0.0,2.0,1.731275,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,2.40,2.40,1,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,0.000000,NaN,NaN,0.000000,0.040445,CAR,0.080000
00-0036439,2020,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0,0.0,0,0.0,0.0,0.0,0.000000,0,6,9,63.0,0,0.0,0.0,79.0,18.0,4.0,2.915174,0,3.200000,0.241825,0.280559,0.559129,0.0,6.30,12.30,5,0.047368,0.051299,0.025388,0.112092,0.049489,0.000000,0.061538,0.055556,0.024745,0.039592,0.331579,0.027577,DEN,0.108000
00-0036442,2020,REG,264,404,2688.0,13,5.0,32.0,231.0,5,3,3429.0,1142.0,150.0,48.966725,0,7.910450,1.200403,37,142.0,3,4.0,1.0,14.0,-0.141221,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.000000,0,0.000000,0.000000,0.000000,0.000000,0.0,173.72,173.72,10,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.190763,CIN,0.973000


In [146]:
snap_df.groupby("player_id").size().sort_values(ascending=False)

player_id
00-0033481    3
00-0019596    2
00-0032775    2
00-0027656    2
00-0036360    2
             ..
00-0033114    1
00-0033118    1
00-0033119    1
00-0033127    1
00-0036456    1
Length: 637, dtype: int64

In [147]:
snap_df[snap_df["player_id"] == "00-0033481"]

,player_id,team,season,snap_pct
279,00-0033481,KC,2020,0.055455
280,00-0033481,MIA,2020,0.000000
281,00-0033481,TB,2020,0.130000


In [152]:
player_stats_df[player_stats_df["player_id"] == "00-0033481"]

,player_id,season,season_type,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr_x,special_teams_tds,fantasy_points,fantasy_points_ppr,games,tgt_sh,ay_sh,yac_sh,wopr_y,ry_sh,rtd_sh,rfd_sh,rtdfd_sh,dom,w8dom,yptmpa,ppr_sh,team,snap_pct
291,00-0033481,2020,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0,1,1,11.0,0,0.0,0.0,1.0,10.0,1.0,0.968359,0,11.0,0.041667,0.006024,0.066717,0.0,1.1,2.1,1,0.04,0.006024,0.076336,0.064819,0.051643,0.0,0.111111,0.1,0.025822,0.041315,0.44,0.025326,KC,0.055455
292,00-0033481,2020,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0,1,1,11.0,0,0.0,0.0,1.0,10.0,1.0,0.968359,0,11.0,0.041667,0.006024,0.066717,0.0,1.1,2.1,1,0.04,0.006024,0.076336,0.064819,0.051643,0.0,0.111111,0.1,0.025822,0.041315,0.44,0.025326,MIA,0.000000
293,00-0033481,2020,REG,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0,0.0,0,0.0,0.0,0.0,0.0,0,1,1,11.0,0,0.0,0.0,1.0,10.0,1.0,0.968359,0,11.0,0.041667,0.006024,0.066717,0.0,1.1,2.1,1,0.04,0.006024,0.076336,0.064819,0.051643,0.0,0.111111,0.1,0.025822,0.041315,0.44,0.025326,TB,0.130000


In [99]:
# missing_pct = (
#     player_stats_df
#     .isna()
#     .mean()
#     .sort_values(ascending=False)
#     .reset_index()
# )
# missing_pct.columns = ["feature", "missing_pct"]
# print(missing_pct.to_string())

### Load Team Targets

In [100]:
# pbp = nfl.import_pbp_data(years)
# pass_plays = pbp[pbp["pass_attempt"] == 1]
# team_targets = (
#     pass_plays.groupby(["posteam", "season"])
#     .size()
#     .reset_index(name="team_target_total")
#     .rename(columns={"posteam": "team"})
# )
# team_targets

### Advanced Stats

In [155]:
for stat_type in ("passing", "rushing", "receiving"):
    try:
        raw = nfl.import_ngs_data(stat_type, years)
        records = _parse_ngs(raw, stat_type)
        # print(records)
        break
    except Exception as e:
        print(e)

In [161]:
ngs_df = nfl.import_ngs_data("rushing", years).rename(columns={"player_gsis_id": "player_id"})
ngs_df

,season,season_type,week,player_display_name,player_position,team_abbr,efficiency,percent_attempts_gte_eight_defenders,avg_time_to_los,rush_attempts,rush_yards,avg_rush_yards,rush_touchdowns,player_id,player_first_name,player_last_name,player_jersey_number,player_short_name,expected_rush_yards,rush_yards_over_expected,rush_yards_over_expected_per_att,rush_pct_over_expected
2356,2020,REG,0,Peyton Barber,RB,WAS,4.389922,42.553191,2.662659,94,258,2.744681,4,00-0032741,Kenneth,Barber,34,P.Barber,332.583707,-74.583707,-0.793444,0.351064
2357,2020,REG,0,Alvin Kamara,RB,NO,3.659195,11.764706,2.754254,187,932,4.983957,16,00-0033906,Alvin,Kamara,41,A.Kamara,841.079108,76.920892,0.415789,0.394595
2358,2020,REG,0,David Johnson,RB,HOU,3.340810,14.965986,2.568261,147,691,4.700680,6,00-0032187,David,Johnson,31,D.Johnson,630.780360,60.219640,0.409657,0.360544
2359,2020,REG,0,Mike Davis,RB,CAR,4.153037,32.727273,2.787275,165,642,3.890909,6,00-0032063,Michael,Davis,28,M.Davis,615.032319,10.967681,0.068122,0.341615
2360,2020,REG,0,Kalen Ballage,RB,LAC,4.503069,19.780220,2.706610,91,303,3.329670,3,00-0034799,Kalen,Ballage,31,K.Ballage,348.162668,-51.162668,-0.574861,0.404494
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2947,2020,POST,20,Darrel Williams,RB,KC,3.858462,23.076923,2.683833,13,52,4.000000,1,00-0034301,Darrel,Williams,31,Darr.Williams,41.602380,5.397620,0.449802,0.416667
2948,2020,POST,20,Leonard Fournette,RB,TB,3.754364,33.333333,2.742636,12,55,4.583333,1,00-0033856,Leonard,Fournette,28,L.Fournette,40.609425,14.390575,1.199215,0.500000
2949,2020,POST,20,Ronald Jones,RB,TB,7.995625,20.000000,2.573222,10,16,1.600000,0,00-0034816,Ronald,Jones,27,R.Jones,32.076809,-16.076809,-1.607681,0.100000
2950,2020,POST,22,Ronald Jones,RB,TB,3.351311,33.333333,2.375917,12,61,5.083333,0,00-0034816,Ronald,Jones,27,R.Jones,60.172482,0.827518,0.068960,0.416667


In [162]:
ngs_df[ngs_df["player_id"] == "00-0033481"]

,season,season_type,week,player_display_name,player_position,team_abbr,efficiency,percent_attempts_gte_eight_defenders,avg_time_to_los,rush_attempts,rush_yards,avg_rush_yards,rush_touchdowns,player_id,player_first_name,player_last_name,player_jersey_number,player_short_name,expected_rush_yards,rush_yards_over_expected,rush_yards_over_expected_per_att,rush_pct_over_expected


### Weekly Snap Counts

In [163]:
snaps_df = nfl.import_snap_counts(years)
snaps_df = player_ids.merge(snaps_df, on=["pfr_player_id"], how="inner")
snaps_df

,player_id,pfr_player_id,game_id,pfr_game_id,season,game_type,week,player,position,team,opponent,offense_snaps,offense_pct,defense_snaps,defense_pct,st_snaps,st_pct
0,00-0036442,BurrJo01,2020_01_LAC_CIN,202009130cin,2020,REG,1,Joe Burrow,QB,CIN,LAC,68.0,1.00,0.0,0.0,0.0,0.0
1,00-0036442,BurrJo01,2020_02_CIN_CLE,202009170cle,2020,REG,2,Joe Burrow,QB,CIN,CLE,92.0,1.00,0.0,0.0,0.0,0.0
2,00-0036442,BurrJo01,2020_03_CIN_PHI,202009270phi,2020,REG,3,Joe Burrow,QB,CIN,PHI,71.0,0.99,0.0,0.0,0.0,0.0
3,00-0036442,BurrJo01,2020_04_JAX_CIN,202010040cin,2020,REG,4,Joe Burrow,QB,CIN,JAX,75.0,1.00,0.0,0.0,0.0,0.0
4,00-0036442,BurrJo01,2020_05_CIN_BAL,202010110rav,2020,REG,5,Joe Burrow,QB,CIN,BAL,67.0,1.00,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20507,00-0022531,PeteJa21,2020_08_DAL_PHI,202011010phi,2020,REG,8,Jason Peters,T,PHI,DAL,64.0,1.00,0.0,0.0,0.0,0.0
20508,00-0022531,PeteJa21,2020_10_PHI_NYG,202011150nyg,2020,REG,10,Jason Peters,T,PHI,NYG,67.0,1.00,0.0,0.0,0.0,0.0
20509,00-0022531,PeteJa21,2020_11_PHI_CLE,202011220cle,2020,REG,11,Jason Peters,T,PHI,CLE,47.0,0.69,0.0,0.0,0.0,0.0
20510,00-0022531,PeteJa21,2020_12_SEA_PHI,202011300phi,2020,REG,12,Jason Peters,T,PHI,SEA,66.0,0.93,0.0,0.0,0.0,0.0


In [164]:
snaps_df[snaps_df["player_id"] == "00-0033481"]

,player_id,pfr_player_id,game_id,pfr_game_id,season,game_type,week,player,position,team,opponent,offense_snaps,offense_pct,defense_snaps,defense_pct,st_snaps,st_pct
11816,00-0033481,KempMa00,2020_01_HOU_KC,202009100kan,2020,REG,1,Marcus Kemp,WR,KC,HOU,0.0,0.00,0.0,0.0,11.0,0.44
11817,00-0033481,KempMa00,2020_02_KC_LAC,202009200sdg,2020,REG,2,Marcus Kemp,WR,KC,LAC,0.0,0.00,0.0,0.0,18.0,0.64
11818,00-0033481,KempMa00,2020_03_KC_BAL,202009280rav,2020,REG,3,Marcus Kemp,WR,KC,BAL,0.0,0.00,0.0,0.0,15.0,0.60
11819,00-0033481,KempMa00,2020_04_NE_KC,202010050kan,2020,REG,4,Marcus Kemp,WR,KC,NE,0.0,0.00,0.0,0.0,11.0,0.48
11820,00-0033481,KempMa00,2020_06_KC_BUF,202010190buf,2020,REG,6,Marcus Kemp,WR,KC,BUF,2.0,0.03,0.0,0.0,12.0,0.48
11821,00-0033481,KempMa00,2020_07_KC_DEN,202010250den,2020,REG,7,Marcus Kemp,WR,KC,DEN,10.0,0.20,0.0,0.0,19.0,0.63
11822,00-0033481,KempMa00,2020_08_NYJ_KC,202011010kan,2020,REG,8,Marcus Kemp,WR,KC,NYJ,12.0,0.18,0.0,0.0,19.0,0.66
11823,00-0033481,KempMa00,2020_12_KC_TB,202011290tam,2020,REG,12,Marcus Kemp,WR,KC,TB,4.0,0.05,0.0,0.0,20.0,0.69
11824,00-0033481,KempMa00,2020_13_DEN_KC,202012060kan,2020,REG,13,Marcus Kemp,WR,KC,DEN,2.0,0.03,0.0,0.0,16.0,0.62
11825,00-0033481,KempMa00,2020_14_KC_MIA,202012130mia,2020,REG,14,Marcus Kemp,WR,KC,MIA,4.0,0.06,0.0,0.0,23.0,0.68


In [122]:
dict(raw.dtypes)

{'player_id': dtype('O'),
 'pfr_player_id': dtype('O'),
 'game_id': dtype('O'),
 'pfr_game_id': dtype('O'),
 'season': dtype('int32'),
 'game_type': dtype('O'),
 'week': dtype('int32'),
 'player': dtype('O'),
 'position': dtype('O'),
 'team': dtype('O'),
 'opponent': dtype('O'),
 'offense_snaps': dtype('float64'),
 'offense_pct': dtype('float64'),
 'defense_snaps': dtype('float64'),
 'defense_pct': dtype('float64'),
 'st_snaps': dtype('float64'),
 'st_pct': dtype('float64')}

### Injuries

In [125]:
raw = nfl.import_injuries(years).rename(columns={"gsis_id": "player_id"})
raw = player_ids.merge(raw, on=["player_id"], how="inner")
raw["season"] = raw["season"].astype("int")
raw

,player_id,pfr_player_id,season,game_type,team,week,position,full_name,first_name,last_name,report_primary_injury,report_secondary_injury,report_status,practice_primary_injury,practice_secondary_injury,practice_status,date_modified
0,00-0036212,TagoTu00,2020,REG,MIA,1.0,QB,Tua Tagovailoa,Tua,Tagovailoa,None,None,None,Hip,None,Full Participation in Practice,2020-09-11 10:36:59+00:00
1,00-0036212,TagoTu00,2020,REG,MIA,4.0,QB,Tua Tagovailoa,Tua,Tagovailoa,Illness,None,Questionable,Illness,None,Limited Participation in Practice,2020-10-02 10:17:38+00:00
2,00-0036212,TagoTu00,2020,REG,MIA,11.0,QB,Tua Tagovailoa,Tua,Tagovailoa,None,None,None,Foot,None,Full Participation in Practice,2020-11-20 11:13:30+00:00
3,00-0036212,TagoTu00,2020,REG,MIA,12.0,QB,Tua Tagovailoa,Tua,Tagovailoa,left Thumb,None,Questionable,left Thumb,None,Limited Participation in Practice,2020-11-27 10:52:27+00:00
4,00-0036212,TagoTu00,2020,REG,MIA,13.0,QB,Tua Tagovailoa,Tua,Tagovailoa,left Thumb,None,Questionable,left Thumb,None,Limited Participation in Practice,2020-12-04 11:30:57+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4611,00-0022531,PeteJa21,2020,REG,PHI,4.0,T,Jason Peters,Jason,Peters,Foot,None,Questionable,Foot,None,Limited Participation in Practice,2020-10-02 12:50:33+00:00
4612,00-0022531,PeteJa21,2020,REG,PHI,10.0,T,Jason Peters,Jason,Peters,None,None,None,Not Injury Related,None,Full Participation in Practice,2020-11-13 12:35:02+00:00
4613,00-0022531,PeteJa21,2020,REG,PHI,11.0,T,Jason Peters,Jason,Peters,None,None,None,Not Injury Related,None,Full Participation in Practice,2020-11-20 12:26:20+00:00
4614,00-0022531,PeteJa21,2020,REG,PHI,12.0,T,Jason Peters,Jason,Peters,Toe,None,Questionable,Toe,None,Limited Participation in Practice,2020-11-28 12:03:25+00:00


In [124]:
dict(raw.dtypes)

{'player_id': dtype('O'),
 'pfr_player_id': dtype('O'),
 'season': dtype('float64'),
 'game_type': dtype('O'),
 'team': dtype('O'),
 'week': dtype('float64'),
 'position': dtype('O'),
 'full_name': dtype('O'),
 'first_name': dtype('O'),
 'last_name': dtype('O'),
 'report_primary_injury': dtype('O'),
 'report_secondary_injury': dtype('O'),
 'report_status': dtype('O'),
 'practice_primary_injury': dtype('O'),
 'practice_secondary_injury': dtype('O'),
 'practice_status': dtype('O'),
 'date_modified': datetime64[ns, UTC]}

### Schedules

In [154]:
schedules_df = nfl.import_schedules(years)

In [155]:
schedules_df

,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,old_game_id,gsis,nfl_detail_id,pfr,pff,espn,ftn,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
5583,2020_01_HOU_KC,2020,REG,1,2020-09-10,Thursday,20:20,HOU,20,KC,34,Home,14,54,0,2020091000,58168,NaN,202009100kan,18532.0,401220225,5325.0,7,7,349.0,-423.0,9.5,-105.0,-105.0,53.5,-109.0,-102.0,0,outdoors,grass,56.0,7.0,00-0033537,00-0033873,Deshaun Watson,Patrick Mahomes,Bill O'Brien,Andy Reid,Clete Blakeman,KAN00,Arrowhead Stadium
5584,2020_01_SEA_ATL,2020,REG,1,2020-09-13,Sunday,13:00,SEA,38,ATL,25,Home,-13,63,0,2020091300,58169,NaN,202009130atl,18533.0,401220313,5334.0,7,7,102.0,-112.0,1.0,-105.0,-105.0,49.5,-108.0,-103.0,0,closed,fieldturf,NaN,NaN,00-0029263,00-0026143,Russell Wilson,Matt Ryan,Pete Carroll,Dan Quinn,Shawn Hochuli,ATL97,Mercedes-Benz Stadium
5585,2020_01_CLE_BAL,2020,REG,1,2020-09-13,Sunday,13:00,CLE,6,BAL,38,Home,32,44,0,2020091301,58170,NaN,202009130rav,18534.0,401220147,5327.0,7,7,260.0,-302.0,7.0,-107.0,-103.0,47.0,100.0,-112.0,1,outdoors,grass,76.0,5.0,00-0034855,00-0034796,Baker Mayfield,Lamar Jackson,Kevin Stefanski,John Harbaugh,Ronald Torbert,BAL00,M&T Bank Stadium
5586,2020_01_NYJ_BUF,2020,REG,1,2020-09-13,Sunday,13:00,NYJ,17,BUF,27,Home,10,44,0,2020091302,58171,NaN,202009130buf,18535.0,401220116,5332.0,7,7,242.0,-279.0,6.5,-110.0,100.0,39.5,-114.0,102.0,1,outdoors,astroturf,67.0,15.0,00-0034869,00-0034857,Sam Darnold,Josh Allen,Adam Gase,Sean McDermott,Shawn Smith,BUF00,New Era Field
5587,2020_01_LV_CAR,2020,REG,1,2020-09-13,Sunday,13:00,LV,34,CAR,30,Home,-4,64,0,2020091303,58172,NaN,202009130car,18536.0,401220370,5330.0,7,7,-124.0,134.0,-3.0,-112.0,-113.0,48.0,-101.0,-110.0,0,outdoors,grass,81.0,5.0,00-0031280,00-0031237,Derek Carr,Teddy Bridgewater,Jon Gruden,Matt Rhule,Brad Allen,CAR00,Bank of America Stadium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5847,2020_19_CLE_KC,2020,DIV,19,2021-01-17,Sunday,15:05,CLE,17,KC,22,Home,5,39,0,2021011700,58497,NaN,202101170kan,19286.0,401220400,5589.0,7,14,316.0,-376.0,7.5,-105.0,-105.0,55.5,-106.0,-105.0,0,outdoors,astroturf,40.0,13.0,00-0034855,00-0033873,Baker Mayfield,Patrick Mahomes,Kevin Stefanski,Andy Reid,Clay Martin,KAN00,Arrowhead Stadium
5848,2020_19_TB_NO,2020,DIV,19,2021-01-17,Sunday,18:40,TB,30,NO,20,Home,-10,50,0,2021011701,58498,NaN,202101170nor,19287.0,401220399,5590.0,8,7,120.0,-133.0,2.5,-104.0,-106.0,53.0,-108.0,-103.0,1,dome,astroturf,NaN,NaN,00-0019596,00-0020531,Tom Brady,Drew Brees,Bruce Arians,Sean Payton,Shawn Hochuli,NOR00,Mercedes-Benz Superdome
5849,2020_20_TB_GB,2020,CON,20,2021-01-24,Sunday,15:05,TB,31,GB,26,Home,-5,57,0,2021012400,58499,NaN,202101240gnb,19288.0,401220402,5591.0,7,8,157.0,-176.0,3.0,108.0,-119.0,53.0,-102.0,-109.0,0,outdoors,grass,29.0,10.0,00-0019596,00-0023459,Tom Brady,Aaron Rodgers,Bruce Arians,Matt LaFleur,Clete Blakeman,GNB00,Lambeau Field
5850,2020_20_BUF_KC,2020,CON,20,2021-01-24,Sunday,18:40,BUF,24,KC,38,Home,14,62,0,2021012401,58500,NaN,202101240kan,19289.0,401220401,5592.0,8,7,142.0,-158.0,3.0,-108.0,-102.0,54.5,102.0,-114.0,0,outdoors,astroturf,40.0,7.0,00-0034857,00-0033873,Josh Allen,Patrick Mahomes,Sean McDermott,Andy Reid,Bill Vinovich,KAN00,Arrowhead Stadium


In [157]:
import nflreadpy as nrp

In [158]:
sched = nrp.load_schedules(years)

In [159]:
sched

game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,old_game_id,gsis,nfl_detail_id,pfr,pff,espn,ftn,away_rest,home_rest,away_moneyline,home_moneyline,spread_line,away_spread_odds,home_spread_odds,total_line,under_odds,over_odds,div_game,roof,surface,temp,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
str,i32,str,i32,str,str,str,str,i32,str,i32,str,i32,i32,i32,str,i32,str,str,i32,str,i32,i32,i32,i32,i32,f64,i32,i32,f64,i32,i32,i32,str,str,i32,i32,str,str,str,str,str,str,str,str,str
"""2020_01_HOU_KC""",2020,"""REG""",1,"""2020-09-10""","""Thursday""","""20:20""","""HOU""",20,"""KC""",34,"""Home""",14,54,0,"""2020091000""",58168,null,"""202009100kan""",18532,"""401220225""",5325,7,7,349,-423,9.5,-105,-105,53.5,-109,-102,0,"""outdoors""","""grass""",56,7,"""00-0033537""","""00-0033873""","""Deshaun Watson""","""Patrick Mahomes""","""Bill O'Brien""","""Andy Reid""","""Clete Blakeman""","""KAN00""","""Arrowhead Stadium"""
"""2020_01_SEA_ATL""",2020,"""REG""",1,"""2020-09-13""","""Sunday""","""13:00""","""SEA""",38,"""ATL""",25,"""Home""",-13,63,0,"""2020091300""",58169,null,"""202009130atl""",18533,"""401220313""",5334,7,7,102,-112,1.0,-105,-105,49.5,-108,-103,0,"""closed""","""fieldturf""",null,null,"""00-0029263""","""00-0026143""","""Russell Wilson""","""Matt Ryan""","""Pete Carroll""","""Dan Quinn""","""Shawn Hochuli""","""ATL97""","""Mercedes-Benz Stadium"""
"""2020_01_CLE_BAL""",2020,"""REG""",1,"""2020-09-13""","""Sunday""","""13:00""","""CLE""",6,"""BAL""",38,"""Home""",32,44,0,"""2020091301""",58170,null,"""202009130rav""",18534,"""401220147""",5327,7,7,260,-302,7.0,-107,-103,47.0,100,-112,1,"""outdoors""","""grass""",76,5,"""00-0034855""","""00-0034796""","""Baker Mayfield""","""Lamar Jackson""","""Kevin Stefanski""","""John Harbaugh""","""Ronald Torbert""","""BAL00""","""M&T Bank Stadium"""
"""2020_01_NYJ_BUF""",2020,"""REG""",1,"""2020-09-13""","""Sunday""","""13:00""","""NYJ""",17,"""BUF""",27,"""Home""",10,44,0,"""2020091302""",58171,null,"""202009130buf""",18535,"""401220116""",5332,7,7,242,-279,6.5,-110,100,39.5,-114,102,1,"""outdoors""","""astroturf""",67,15,"""00-0034869""","""00-0034857""","""Sam Darnold""","""Josh Allen""","""Adam Gase""","""Sean McDermott""","""Shawn Smith""","""BUF00""","""New Era Field"""
"""2020_01_LV_CAR""",2020,"""REG""",1,"""2020-09-13""","""Sunday""","""13:00""","""LV""",34,"""CAR""",30,"""Home""",-4,64,0,"""2020091303""",58172,null,"""202009130car""",18536,"""401220370""",5330,7,7,-124,134,-3.0,-112,-113,48.0,-101,-110,0,"""outdoors""","""grass""",81,5,"""00-0031280""","""00-0031237""","""Derek Carr""","""Teddy Bridgewater""","""Jon Gruden""","""Matt Rhule""","""Brad Allen""","""CAR00""","""Bank of America Stadium"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2020_19_CLE_KC""",2020,"""DIV""",19,"""2021-01-17""","""Sunday""","""15:05""","""CLE""",17,"""KC""",22,"""Home""",5,39,0,"""2021011700""",58497,null,"""202101170kan""",19286,"""401220400""",5589,7,14,316,-376,7.5,-105,-105,55.5,-106,-105,0,"""outdoors""","""astroturf""",40,13,"""00-0034855""","""00-0033873""","""Baker Mayfield""","""Patrick Mahomes""","""Kevin Stefanski""","""Andy Reid""","""Clay Martin""","""KAN00""","""Arrowhead Stadium"""
"""2020_19_TB_NO""",2020,"""DIV""",19,"""2021-01-17""","""Sunday""","""18:40""","""TB""",30,"""NO""",20,"""Home""",-10,50,0,"""2021011701""",58498,null,"""202101170nor""",19287,"""401220399""",5590,8,7,120,-133,2.5,-104,-106,53.0,-108,-103,1,"""dome""","""astroturf""",null,null,"""00-0019596""","""00-0020531""","""Tom Brady""","""Drew Brees""","""Bruce Arians""","""Sean Payton""","""Shawn Hochuli""","""NOR00""","""Mercedes-Benz Superdome"""
"""2020_20_TB_GB""",2020,"""CON""",20,"""2021-01-24""","""Sunday""","""15:05""","""TB""",31,"""GB""",26,"""Home""",-5,57,0,"""2021012400""",58499,null,"""202101240gnb""",19288,"

### NFL Players

In [4]:
players = nfl.import_players()

In [5]:
players

,gsis_id,display_name,common_first_name,first_name,last_name,short_name,football_name,suffix,esb_id,nfl_id,pfr_id,pff_id,otc_id,espn_id,smart_id,birth_date,position_group,position,ngs_position_group,ngs_position,height,weight,headshot,college_name,college_conference,jersey_number,rookie_season,last_season,latest_team,status,ngs_status,ngs_status_short_description,years_of_experience,pff_position,pff_status,draft_year,draft_round,draft_pick,draft_team
0,00-0028830,Isaako Aaitui,Isaako,Isaako,Aaitui,None,None,None,AAI622937,None,AaitIs00,6998,2535,14856,32004141-4962-2937-61ff-017b1804dec6,1987-01-25,DL,NT,None,None,76.0,307.0,https://static.www.nfl.com/image/private/f_aut...,UNLV,None,0,2011,2014,WAS,DEV,None,None,2,DI,None,NaN,NaN,NaN,None
1,00-0038389,Israel Abanikanda,Israel,Israel,Abanikanda,I.Abanikanda,Israel,None,ABA159567,56008,AbanIs00,122999,10967,4429202,32004142-4115-9567-2e24-0eab29f6a4b9,2002-10-05,RB,RB,RB,RB,70.0,216.0,https://static.www.nfl.com/image/upload/f_auto...,Pittsburgh,Atlantic Coast Conference,None,2023,2026,DAL,ACT,RES,Reserve/Future,3,HB,A,2023.0,5.0,143.0,NYJ
2,00-0024644,Jon Abbate,Jon,Jon,Abbate,None,None,None,ABB051371,None,None,None,None,None,32004142-4205-1371-db95-1abc96313b69,1985-06-18,LB,LB,None,None,71.0,245.0,https://static.www.nfl.com/image/private/f_aut...,Wake Forest,None,67,2007,2007,HOU,RES,None,None,0,None,None,NaN,NaN,NaN,None
3,ABB498348,Vince Abbott,Vince,Vincent,Abbott,None,None,None,ABB498348,None,abbotvin01,None,None,None,32004142-4249-8348-e00f-5fbbe6a0c73c,1958-05-31,SPEC,K,None,None,71.0,207.0,https://static.www.nfl.com/image/private/f_aut...,California State-Fullerton; Washington,None,0,1987,1988,LAC,ACT,None,None,2,None,None,NaN,NaN,NaN,None
4,00-0031021,Jared Abbrederis,Jared,Jared,Abbrederis,J.Abbrederis,Jared,None,ABB650964,41405,AbbrJa00,8811,3115,16836,32004142-4265-0964-fc36-bb0ad76ff6e6,1990-12-17,WR,WR,WR,WR,73.0,195.0,https://static.www.nfl.com/image/private/f_aut...,Wisconsin,None,10,2014,2017,DET,CUT,CUT,None,4,WR,None,2014.0,5.0,176.0,GB
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24403,00-0034508,Gerhard de Beer,Gerhard,Gerhard,de Beer,G.de Beer,Gerhard,None,DEB150981,46692,deBeGe00,46960,7212,3056440,32004445-4215-0981-d8fc-08384330fa6a,1994-07-05,OL,OT,None,None,78.0,312.0,https://static.www.nfl.com/image/private/f_aut...,Arizona,None,0,2018,2018,GB,DEV,CUT,None,0,None,None,NaN,NaN,NaN,None
24404,DEB622935,Case deBruijn,Case,Case,deBruijn,None,None,None,DEB622935,None,deBrCa20,None,None,None,32004445-4262-2935-c44d-416ca8a4c116,1960-04-11,SPEC,P,None,None,72.0,176.0,https://static.www.nfl.com/image/private/f_aut...,Idaho State,None,0,1982,1982,KC,ACT,None,None,0,None,None,1982.0,8.0,214.0,KC
24405,VAN516304,Mark van Eeghen,Mark,Mark,van Eeghen,None,None,None,VAN516304,None,VanEMa00,None,None,None,32005641-4e51-6304-9ea7-e13b6d636311,1952-04-19,RB,RB,None,None,74.0,223.0,https://static.www.nfl.com/image/private/f_aut...,Colgate,None,0,1974,1983,NE,ACT,None,None,10,None,None,1974.0,3.0,75.0,LV
24406,00-0016956,Kimo von Oelhoffen,Kimo,Kimo,von Oelhoffen,None,None,None,VON221488,None,vonOKi20,None,None,565,3200564f-4e22-1488-f980-8c2faa904183,1971-01-30,DL,DT,None,None,76.0,299.0,https://static.www.nfl.com/image/private/f_aut...,Boise State; Hawaii; Walla Walla Community Col...,None,66,1994,2007,PHI,ACT,None,None,14,None,None,1994.0,6.0,162.0,CIN


In [6]:
missing_pct = (
    players
    .isna()
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
missing_pct.columns = ["feature", "missing_pct"]
# print("\nFeature missingness (top 10):")
# print("\nNFL Feature missingness:")
# print(missing_pct.head(10).to_string())
print(missing_pct.to_string())

                         feature  missing_pct
0                         suffix     0.993281
1                     pff_status     0.841978
2                   ngs_position     0.796665
3             ngs_position_group     0.793060
4             college_conference     0.746681
5   ngs_status_short_description     0.745534
6                     short_name     0.698869
7                   pff_position     0.695837
8                  football_name     0.679122
9                     ngs_status     0.679081
10                        otc_id     0.617461
11                        pff_id     0.539331
12                        nfl_id     0.531301
13                    draft_pick     0.499017
14                   draft_round     0.499017
15                    draft_year     0.499017
16                    draft_team     0.499017
17                       espn_id     0.339643
18                        pfr_id     0.089930
19                 jersey_number     0.057522
20                      headshot  

## Rosters

In [194]:
rosters = nfl.import_seasonal_rosters(years)
rosters

,season,team,position,depth_chart_position,jersey_number,status,player_name,first_name,last_name,birth_date,height,weight,college,player_id,espn_id,sportradar_id,yahoo_id,rotowire_id,pff_id,pfr_id,fantasy_data_id,sleeper_id,years_exp,headshot_url,ngs_position,week,game_type,status_description_abbr,football_name,esb_id,gsis_it_id,smart_id,entry_year,rookie_year,draft_club,draft_number,age
0,2020,IND,K,K,4.0,UFA,Adam Vinatieri,Adam,Vinatieri,1972-12-28,72.0,212,South Dakota State,00-0016919,1097,9ecf8040-10f9-4a5c-92da-1b4d77bd6760,3727,395,226,None,3258,120,24.0,https://static.www.nfl.com/image/private/f_aut...,None,1,REG,None,Adam,VIN196019,21213,32005649-4e19-6019-e626-0b58f9aa81e1,1996.0,1996.0,None,NaN,47.0
1,2020,TB,QB,QB,12.0,ACT,Tom Brady,Tom,Brady,1977-08-03,76.0,225,Michigan,00-0019596,2330,41c44740-d0f6-44ab-8347-3b5d515e5ecf,5228,1350,698,BradTo00,4314,167,20.0,https://static.www.nfl.com/image/private/f_aut...,QB,21,SB,A01,Tom,BRA371156,25511,32004252-4137-1156-7ed0-8b9e44948f13,2000.0,2000.0,NE,199.0,43.0
2,2020,NO,QB,QB,9.0,ACT,Drew Brees,Drew,Brees,1979-01-15,72.0,209,Purdue,00-0020531,2580,bb5957e6-ce7d-47ab-8036-22191ffc1c44,5479,2178,802,BreeDr00,7242,289,19.0,https://static.www.nfl.com/image/private/f_aut...,QB,19,DIV,A01,Drew,BRE229498,26250,32004252-4522-9498-de45-55dc89e02748,2001.0,2001.0,SD,32.0,41.0
3,2020,HOU,QB,QB,3.0,INA,Josh McCown,Joshua,McCown,1979-07-04,76.0,218,Sam Houston State,00-0021206,3609,a261fd0a-b096-4db7-bd51-2f13bde485da,5967,2513,1085,McCoJo01,5282,208,18.0,https://static.www.nfl.com/image/private/f_aut...,None,17,REG,A01,Josh,MCC600777,27213,32004d43-4360-0777-1931-84279757cab3,2002.0,2002.0,ARI,81.0,41.0
4,2020,LV,TE,TE,82.0,ACT,Jason Witten,Christopher,Witten,1982-05-06,78.0,263,Tennessee,00-0022127,4527,e38c9b1b-7c51-48a2-ac1d-a752502e8930,6405,3086,1384,WittJa00,722,23,17.0,https://static.www.nfl.com/image/private/f_aut...,TE,17,REG,A01,Jason,WIT559021,28103,32005749-5455-9021-9f23-668d83c1b782,2003.0,2003.0,DAL,69.0,38.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3062,2020,NYJ,RB,RB,0.0,DEV,Pete Guerriero,Peter,Guerriero,NaT,70.0,190,"Monmouth, N.J.",00-0036454,None,None,None,None,None,None,None,None,0.0,None,None,17,REG,P01,Pete,GUE579681,53153,32004755-4557-9681-5e3d-81fdd9ca979d,2020.0,2020.0,None,NaN,NaN
3063,2020,GB,TE,TE,49.0,ACT,Dominique Dafney,Dominique,Dafney,1997-06-03,75.0,235,Indiana State,00-0036456,4036129,921d49c2-64d1-4108-a3a3-0c237e17748a,33367,15181,62825,DafnDo00,22477,7502,0.0,None,TE,20,CON,A01,Dominique,DAF632824,53172,32004441-4663-2824-ae6e-e3f23d8bb792,2020.0,2020.0,None,NaN,23.0
3064,2020,BAL,RB,RB,34.0,DEV,Ty'Son Williams,Ty'Son,Williams,1996-09-04,72.0,220,Brigham Young,00-0036457,3895827,9e31f935-1e60-437d-9859-e1f2bb936aa5,33006,14404,38287,WillTy01,22126,7098,0.0,None,None,19,DIV,P01,Ty'Son,WIL549673,53180,32005749-4c54-9673-99c6-3057efe2b9a6,2020.0,2020.0,None,NaN,23.0
3065,2020,DAL,P,P,1.0,ACT,Hunter Niswander,James,Niswander,1994-11-26,77.0,243,Northwestern,00-0036458,3045246,48efc038-96b7-4c63-a33c-41f6abe6d3de,None,15206,29601,None,None,7516,0.0,None,None,17,REG,A01,Hunter,NIS764137,50975,32004e49-5376-4137-91d4-44cb1a0c721e,2020.0,2020.0,None,NaN,25.0


## Team Context

In [215]:
def _safe_int(val):
    try:
        if pd.isna(val):
            return None
        return int(float(val))
    except (TypeError, ValueError):
        return None
        
def _count_games_per_team(season_pbp: pd.DataFrame) -> dict[str, int]:
    """Count distinct game_ids per team — needed for per-game rate calculations."""
    if "game_id" not in season_pbp.columns:
        return {}
    return season_pbp.groupby("posteam")["game_id"].nunique().to_dict()
 
 
def _compute_points_per_game(schedules: pd.DataFrame, season: int) -> dict[str, float]:
    """
    Compute average points scored per game from the schedules table.
    Unpivots home/away into one row per team per game, then averages.
    """
    season_sched = schedules[
        (schedules["season"] == season) &
        (schedules.get("game_type", pd.Series(["REG"] * len(schedules))) == "REG")
    ].copy()
    required = {"home_team", "away_team", "home_score", "away_score"}
    if season_sched.empty or not required.issubset(season_sched.columns):
        return {}
    home = season_sched[["home_team", "home_score"]].rename(columns={"home_team": "team", "home_score": "points"})
    away = season_sched[["away_team", "away_score"]].rename(columns={"away_team": "team", "away_score": "points"})
    all_games = pd.concat([home, away], ignore_index=True).dropna(subset=["points"])
    return all_games.groupby("team")["points"].mean().round(2).to_dict()
 
 
def _compute_ol_rank(season_pbp: pd.DataFrame) -> dict[str, int]:
    """
    Rank all 32 teams 1–32 on OL quality using sacks allowed per pass attempt.
    Rank 1 = best OL (fewest sacks). Correlates with PFF OL grade at r≈0.65.
    """
    if season_pbp.empty or "sack" not in season_pbp.columns:
        return {}
    pass_plays = season_pbp[season_pbp["pass_attempt"] == 1].copy()
    stats = (
        pass_plays.groupby("posteam")
        .agg(pass_attempts=("pass_attempt", "sum"), sacks=("sack", "sum"))
        .reset_index()
    )
    stats = stats[stats["pass_attempts"] > 0].copy()
    stats["sack_rate"] = stats["sacks"] / stats["pass_attempts"]
    stats["ol_rank"] = stats["sack_rate"].rank(method="min").astype(int)
    return stats.set_index("posteam")["ol_rank"].to_dict()

def ingest_team_context(years: list[int]) -> None:
        """
        Build and store one NFLTeam row per (team, season).
 
        Fields derived from play-by-play (computed here):
          plays_per_game, pass_rate, pass_rate_neutral, team_pass_yards,
          team_rush_yards, team_pass_attempts, team_targets, offensive_line_rank
 
        Fields derived from schedules (computed here):
          points_per_game
 
        Fields from static COACHING_DATA dict (maintained manually):
          head_coach, offensive_coordinator, defensive_coordinator, offensive_scheme
 
        WHY play-by-play for pass rate (not seasonal_data):
          Neutral pass rate — pass rate when score is within one possession —
          is a far better proxy for offensive philosophy than raw pass rate,
          which inflates when teams trail. We can only compute this from pbp
          with score differential filtering.
 
        WHY sack count for OL rank:
          True OL grades (PFF) are paywalled. Sacks allowed per pass attempt
          correlates with PFF OL grade at r≈0.65 historically. We rank teams
          1–32 (1=best OL) by inverse sack rate as an open-source proxy.
        """
        print(f"Ingesting team context for {years}...")
 
        pbp_cols = [
            "season", "week", "game_id", "posteam", "play_type",
            "pass_attempt", "rush_attempt", "sack", "score_differential",
            "passing_yards", "rushing_yards", "half_seconds_remaining",
        ]
        try:
            pbp = nfl.import_pbp_data(years=years, columns=pbp_cols)
        except Exception as e:
            print(f"Failed to load play-by-play for team context: {e}")
            return
 
        pbp = pbp[pbp["play_type"].isin(["pass", "run", "qb_kneel", "qb_spike"])].copy()
        pbp = pbp[pbp["posteam"].notna()].copy()
 
        try:
            schedules = nfl.import_schedules(years)
        except Exception as e:
            print(f"Could not load schedules (points_per_game will be null): {e}")
            schedules = pd.DataFrame()
 
        try:
            team_desc = nfl.import_team_desc()
            team_name_map = (
                team_desc.set_index("team_abbr")["team_name"].to_dict()
                if "team_abbr" in team_desc.columns else {}
            )
        except Exception:
            team_name_map = {}
 
        records = []
        teams = pbp["posteam"].dropna().unique()
 
        for season in years:
            season_pbp = pbp[pbp["season"] == season].copy()
            games_per_team = _count_games_per_team(season_pbp)
            ppg_map = _compute_points_per_game(schedules, season) if not schedules.empty else {}
            ol_ranks = _compute_ol_rank(season_pbp)
 
            for team in teams:
                team_pbp = season_pbp[season_pbp["posteam"] == team]
                if team_pbp.empty:
                    continue
 
                n_games = games_per_team.get(team, 1)
                pass_plays = team_pbp[team_pbp["pass_attempt"] == 1]
                rush_plays = team_pbp[team_pbp["rush_attempt"] == 1]
                total_plays = len(pass_plays) + len(rush_plays)
                pass_rate = len(pass_plays) / total_plays if total_plays > 0 else None
 
                # Neutral script: score within 7pts AND not final 2 min of half
                neutral_mask = (
                    (team_pbp["score_differential"].abs() <= 7) &
                    (team_pbp["half_seconds_remaining"] > 120)
                )
                neutral = team_pbp[neutral_mask]
                neutral_total = neutral["pass_attempt"].sum() + neutral["rush_attempt"].sum()
                pass_rate_neutral = (
                    float(neutral["pass_attempt"].sum() / neutral_total)
                    if neutral_total > 0 else pass_rate
                )
 
                # Targets = pass attempts excluding sacks
                target_plays = pass_plays[pass_plays["sack"] != 1]
 
                coaching = COACHING_DATA.get(
                    (team, season),
                    COACHING_DATA.get((team, season - 1), _COACHING_FALLBACK)
                )
 
                records.append(dict(
                    team_abbr=team,
                    season=season,
                    full_name=team_name_map.get(team) or TEAM_FULL_NAMES.get(team) or team,
                    head_coach=coaching["head_coach"],
                    offensive_coordinator=coaching["offensive_coordinator"],
                    defensive_coordinator=coaching["defensive_coordinator"],
                    offensive_scheme=coaching["offensive_scheme"],
                    plays_per_game=round(len(team_pbp) / n_games, 2),
                    pass_rate=round(pass_rate, 4) if pass_rate is not None else None,
                    pass_rate_neutral=round(pass_rate_neutral, 4) if pass_rate_neutral is not None else None,
                    team_pass_yards=_safe_int(pass_plays["passing_yards"].sum()),
                    team_rush_yards=_safe_int(rush_plays["rushing_yards"].sum()),
                    team_pass_attempts=len(pass_plays),
                    team_targets=len(target_plays),
                    points_per_game=ppg_map.get(team),
                    offensive_line_rank=ol_ranks.get(team),
                ))
        return pd.DataFrame(records)
        # with get_session(self.engine) as session:
        #     for record in records:
        #         session.merge(record)
        #     session.commit()
 
        # logger.info(f"  → {len(records)} team-season rows written to nfl_teams.")

In [216]:
team_context_df = ingest_team_context(years)

Ingesting team context for [2020]...
2020 done.
Downcasting floats.


In [217]:
team_context_df

,team_abbr,season,full_name,head_coach,offensive_coordinator,defensive_coordinator,offensive_scheme,plays_per_game,pass_rate,pass_rate_neutral,team_pass_yards,team_rush_yards,team_pass_attempts,team_targets,points_per_game,offensive_line_rank
0,SF,2020,San Francisco 49ers,Kyle Shanahan,Mike LaFleur,Robert Saleh,Spread RPO,65.62,0.5838,0.5148,4320,1889,613,574,23.50,19
1,ARI,2020,Arizona Cardinals,Kliff Kingsbury,Tom Clements,Vance Joseph,Air Raid,67.88,0.5580,0.5417,4102,2237,606,577,25.62,10
2,CHI,2020,Chicago Bears,Matt Nagy,Bill Lazor,Chuck Pagano,West Coast,64.41,0.6228,0.5441,4124,1695,682,645,23.25,15
3,DET,2020,Detroit Lions,Matt Patricia,Darrell Bevell,Cory Undlin,West Coast,62.19,0.6312,0.5601,4397,1499,628,586,23.56,22
4,CLE,2020,Cleveland Browns,Kevin Stefanski,Alex Van Pelt,Joe Woods,West Coast,63.78,0.5226,0.5195,4168,2611,600,573,25.50,6
5,BAL,2020,Baltimore Ravens,John Harbaugh,Greg Roman,Don Martindale,Run Heavy,62.89,0.4496,0.4180,3320,3456,509,468,29.25,28
6,LA,2020,Los Angeles Rams,Sean McVay,Kevin O'Connell,Brandon Staley,Spread RPO,67.28,0.5582,0.5814,4541,2278,676,644,23.25,8
7,DAL,2020,Dallas Cowboys,Mike McCarthy,Kellen Moore,Mike Nolan,West Coast,69.94,0.6122,0.5404,4511,1788,685,641,24.69,20
8,GB,2020,Green Bay Packers,Matt LaFleur,Nathaniel Hackett,Mike Pettine,West Coast,63.17,0.5620,0.5750,4941,2373,639,613,31.81,5
9,MIN,2020,Minnesota Vikings,Mike Zimmer,None,Andre Patterson,West Coast,64.62,0.5435,0.4907,4265,2283,562,523,26.88,23
